<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-13-The-Minimal-Admissibility-Core/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 13 — The Minimal Admissibility Core (Master Reference)
## Reviewer Walkthrough · Phase 22 Edition

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18614663.svg)](https://doi.org/10.5281/zenodo.18614663)

**Author:** E.S. Brooke · **Paper:** v8.6 (70pp) · **Codebase:** v6.9

---

### What this notebook is

Paper 13 is the **master reference**. It indexes, tags, and score-keeps the entire APF theorem bank (420 theorems / 437 verify_all checks / 34 modules / 48 predictions / 0 free parameters).

### What this notebook is **not**

A self-contained argument (it assumes Papers 0-8). A proof source (it indexes proofs; cite the actual location). A static document (the scorecard ticks forward with each codebase update).

### Before you begin

If you are a cold AI agent or human reviewer new to APF, read these four files in `ai_context/` **first**:

1. **`ARGUMENT_FLOW.md`** — the one-page structural spine.
2. **`LOCAL_VS_IMPORTED.md`** — what this paper proves vs imports.
3. **`CLAIMS_LEDGER.md`** — row-by-row attack surface.
4. **`DO_NOT_CLAIM.md`** — predictable overclaims and how to avoid them.


## §1 · Setup

Clone the paper-companion repo and load the rendering helpers.

In [ ]:
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-13-The-Minimal-Admissibility-Core.git 2>/dev/null || true
%cd -q APF-Paper-13-The-Minimal-Admissibility-Core
!pip install -q -e . 2>&1 | tail -2

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from apf.apf_utils import dag_reset, dag_put, dag_has, dag_get
from fractions import Fraction
from IPython.display import display, Markdown, Latex, HTML
import inspect

print(f'Bank registry loaded: {len(REGISTRY)} checks.')
print('This repo: 340 bank-registered checks in full-codebase mode — the largest bundle.')

### Phase 22 `show()` helper

Every check returns a result dict. `show()` runs the check, badges the status colour-coded by epistemic tag, and surfaces the mathematical content inline.

Colour code: 🟢 `[P]` proved from A1 · 🟡 `[P_structural]` conditional on upstream · 🟣 `[P_arith]` arithmetic identity · 🔵 `[P+lattice]` lattice QCD input · 🟠 `[C]` conjecture · 🔴 `[FAIL]` check did not pass.

In [ ]:

def _epistemic_badge(tag):
    colors = {
        'P': ('🟢', '#2ecc71', 'proved from A1'),
        'P_structural': ('🟡', '#f1c40f', 'conditional on upstream derivation'),
        'P_arith': ('🟣', '#9b59b6', 'arithmetic identity once formula chosen'),
        'P+lattice': ('🔵', '#3498db', 'proved using lattice QCD input'),
        'C': ('🟠', '#e67e22', 'conjecture; open, flagged'),
    }
    emoji, col, explain = colors.get(str(tag).strip('[]'), ('⚪', '#7f8c8d', 'unknown'))
    return f'<span style="background:{col}22;border:1px solid {col};border-radius:4px;padding:2px 8px;color:{col};font-weight:600">{emoji} {tag}</span> <em style="color:#666;font-size:0.9em">{explain}</em>'


def _render_value(v):
    if isinstance(v, Fraction):
        return f'$\\displaystyle \\frac{{{v.numerator}}}{{{v.denominator}}} = {float(v):.6f}$'
    if isinstance(v, dict) and all(isinstance(val, Fraction) for val in v.values()):
        items = [f'{k}={frac.numerator}/{frac.denominator}' for k, frac in v.items()]
        return '$' + ',\\ '.join(items) + '$'
    if isinstance(v, float):
        return f'`{v:.9g}`'
    if isinstance(v, (list, tuple)) and len(v) < 8:
        return '`' + ', '.join(str(x) for x in v) + '`'
    return f'`{v}`'


def show(check_name, *, run=True, verbose=True):
    try:
        check = get_check(check_name)
    except KeyError:
        display(Markdown(f'**❓ Check `{check_name}` not found.**'))
        return None

    display(Markdown(f'#### `{check_name}`'))
    doc = (check.__doc__ or '').strip()
    first_line = doc.split('\n')[0] if doc else '(no docstring)'
    display(Markdown(f'**Statement:** {first_line}'))

    if not run:
        return None

    try:
        result = check()
        passed = True
    except Exception as e:
        result = {'error': f'{type(e).__name__}: {e}'}
        passed = False

    tag = 'P'
    if isinstance(result, dict):
        for k in ('epistemic_status', 'epistemic', 'tag'):
            if k in result:
                tag = result[k]
                break
    if not passed:
        display(Markdown(f'**Status:** <span style="color:#e74c3c;font-weight:700">🔴 [FAIL]</span>'))
    else:
        display(Markdown(f'**Status:** {_epistemic_badge(tag)}'))

    if isinstance(result, dict):
        if 'key_result' in result:
            display(Markdown(f'**Key result:** {_render_value(result["key_result"])}'))
        if 'error' in result:
            display(Markdown(f'**Error:** `{result["error"]}`'))

        if verbose:
            skip = {'key_result', 'name', 'epistemic', 'epistemic_status', 'tag',
                    'dependencies', 'cross_refs', 'error', 'artifacts', 'statement',
                    'identity', 'consistent'}
            extra = {k: v for k, v in result.items() if k not in skip}
            if extra:
                rows = []
                for k, v in list(extra.items())[:10]:
                    rows.append(f'| `{k}` | {_render_value(v)} |')
                if rows:
                    display(Markdown('**Fields surfaced by the check:**\n\n| Field | Value |\n|---|---|\n' + '\n'.join(rows)))

        if 'dependencies' in result and result['dependencies']:
            deps = result['dependencies']
            if isinstance(deps, (list, tuple)):
                deps_str = ' · '.join(f'`{d}`' for d in deps)
                display(Markdown(f'**Depends on:** {deps_str}'))

    return result


print('show() helper loaded. Phase 22 gorgeous-math rendering enabled.')


## §2 · The scorecard

$$\text{APF Codebase v6.9} \;=\; \begin{cases}
\textbf{437} & \text{verify\_all checks (all pass)}\\
\textbf{420} & \text{bank-registered theorems}\\
\textbf{34} & \text{registered modules} + \texttt{apf/standalone/}\\
\textbf{48} & \text{quantitative predictions}\\
\textbf{32/39} & \text{tested within 3σ (mean error 3.83\%, median 0.37\%)}\\
\textbf{0} & \text{free parameters}
\end{cases}$$

Always verify against `EXPECTED_THEOREM_COUNT` in `apf/bank.py` before citing.

## §3 · Four load-bearing theorems

If you only read four theorems, read these:

1. **$L_{\rm gauge\_template\_uniqueness}$** (Paper 1) — SU(3)×SU(2)×U(1) template is forced.
2. **$T_{\rm field}$** (Papers 1+4) — 1-of-4680 fermion filter; 45 fermions survive.
3. **$T_{11}$** (Paper 6) — $\Omega_\Lambda = 42/61$ with $C_{\rm vacuum} = 42 = 27 + 3 + 12$.
4. **$T_{\rm ACC\_unification}$** (Paper 8 Theorem 1.1) — gauge-cosmological bridge at the ledger level.

In [ ]:
show('check_L_gauge_template_uniqueness')

In [ ]:
show('check_T_field')

In [ ]:
show('check_T11')

In [ ]:
show('check_T_ACC_unification')

## §4 · Derivation chain (one line)

$$\text{A1}
\to L_{\rm nc}, L_{\rm irr}, L_{\rm col}
\to L_{\rm gauge\_template\_uniqueness}
\to T_{\rm gauge}
\to T_{\rm field}
\to L_{\rm count}
\to \text{all 25 Layer-II predictions}$$

Plus the Cauchy branch:

$$\text{A1}
\to L_{\rm Cauchy\_uniqueness}\;(F(d)=d)
\to \gamma = 17/4
\to \sin^2\theta_W = 3/13$$

## §5 · Module architecture

34 modules + `apf/standalone/`. Key modules:

- **`core.py`** — A1, PLEC, $L_\varepsilon^*$.
- **`gauge.py`** — Theorem R + gauge template + $L_{\rm count}$.
- **`gravity.py`** — $T_{11}$ ($C_{\rm vacuum}=42$), $L_{\rm self\_exclusion}$ ($d_{\rm eff}=102$), bridge theorem.
- **`unification.py`** — ACC record, six projections, four identities, $T_{\rm ACC\_unification}$.
- **`unification_three_levels.py`** — integer/scalar/subspace refinement of I1-I4.
- **`subspace_functors.py`** — F_horizon / F_quantum / F_operator.
- **`fractional_reading.py`** — FRE + $\Omega_\Lambda = 42/61$ = $S(V_\Lambda)/S_{\rm SM}$.
- **`lambda_absolute.py`** — $\rho_\Lambda / M_{\rm Pl}^4 = 42/102^{62}$ (with `[C]` on structural coefficient).
- **`crystal.py` + `crystal_metrics.py`** — Enforcement Crystal walker + analytical metrics.


## §6 · Full bank pass

Run every check in this repo's bundled codebase subset.

In [ ]:
from apf.bank import run_all
from collections import Counter
results = run_all()
outcomes = Counter()
for name, res in results.items():
    if isinstance(res, dict) and 'error' in res:
        outcomes['ERROR'] += 1
    else:
        outcomes['PASS'] += 1
print(f'Total: {len(results)} checks')
for status, n in outcomes.most_common():
    print(f'  {status}: {n}')

### Where to go next

- **Paper PDF** — the main paper + Technical Supplement in this repo.
- **`ai_context/`** — the four audit-native files (ARGUMENT_FLOW, LOCAL_VS_IMPORTED, CLAIMS_LEDGER, DO_NOT_CLAIM).
- **[Canonical codebase v6.9](https://doi.org/10.5281/zenodo.18604548)** — the full 420-theorem bank.
- **[Paper 8 companion repo](https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger)** — the pilot implementation of Phase 22 (anti-smuggling tests + minimal working example + full gorgeous-math Colab).

### Citation

```bibtex
@software{Brooke_Paper13_2026,
  author  = {Brooke, Ethan S.},
  title   = {The Minimal Admissibility Core},
  year    = 2026,
  version = {v8.6 (70pp)},
  doi     = {10.5281/zenodo.18614663}
}
```

*Paper-companion repo · v8.6 (70pp) · Phase 22 gorgeous-math edition · 2026-04-24.*